In [ ]:
# ============ DOC-SIDE REWRITING (PILOT — 50 abstracts) ============
import os, re, json, requests
from datetime import datetime
from tqdm import tqdm
import torch
import numpy as np
import torch.nn.functional as F
from torch import Tensor
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset, Dataset

# Path fix
def find_cache_dir():
    for c in ['baseline/cache', 'cache', '../baseline/cache']:
        if os.path.exists(c): return c
    raise FileNotFoundError()
CACHE_DIR = find_cache_dir()
RESULTS_DIR = os.path.join(os.path.dirname(CACHE_DIR), 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

# Setup
device = 'mps' if torch.backends.mps.is_available() else 'cpu'
batch_size = 16
print(f"Device: {device}")

# Load corpus
collection_data = load_dataset("sschellhammer/CT26_Task1_SourceRetrievalForScientificWebClaims", "collection")["collection"]
print(f"Collection: {len(collection_data)} abstracts")

# Pilot: ilk 50 abstract'ı al
PILOT_N = 50
pilot = collection_data.select(range(PILOT_N))

# Ollama prompt
OLLAMA_URL = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "qwen2.5:1.5b-instruct"

def rewrite_to_tweet(title, abstract, retries=2):
    prompt = (
        "Rewrite the following scientific paper into ONE tweet-like sentence (max 280 chars). "
        "Capture the main finding/claim in informal but accurate language, like a science communicator. "
        "Output ONLY the tweet, no preamble.\n\n"
        f"Title: {title}\n"
        f"Abstract: {abstract[:1500]}\n\n"
        "Tweet:"
    )
    payload = {
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "stream": False,
        "keep_alive": "60m",
        "options": {"temperature": 0.0, "num_predict": 100}
    }
    for attempt in range(retries + 1):
        try:
            r = requests.post(OLLAMA_URL, json=payload, timeout=60)
            r.raise_for_status()
            return r.json()["response"].strip().split("\n")[0]
        except Exception:
            if attempt == retries:
                return f"{title}. {abstract[:200]}"
            continue

# Pilot run
print(f"\nPilot: rewriting {PILOT_N} abstracts...")
rewrites = []
for ex in tqdm(pilot, desc="Rewriting"):
    rewrite = rewrite_to_tweet(ex["title"], ex["abstract"])
    rewrites.append(rewrite)

# Show samples
print("\n=== SAMPLE REWRITES ===\n")
for i in range(min(8, PILOT_N)):
    title = pilot[i]["title"][:100]
    abstract = pilot[i]["abstract"][:150]
    rewrite = rewrites[i][:200]
    print(f"[{i}] TITLE:    {title}")
    print(f"    ABSTRACT: {abstract}...")
    print(f"    TWEET:    {rewrite}")
    print()

# Save pilot
with open(os.path.join(CACHE_DIR, 'pilot_doc_rewrites.json'), 'w', encoding='utf-8') as f:
    json.dump({
        "pubkeys": [pilot[i]["pubkey"] for i in range(PILOT_N)],
        "rewrites": rewrites
    }, f, ensure_ascii=False, indent=2)

print(f"\nPilot saved. Speed: ~{(50/4)*60:.0f}s for 50 abs → estimated full corpus: {(10000/PILOT_N) * 4:.0f} min")